# LAB-HW-10 — AXI / Burst Measurement

**今天只新增一件事：** 让 programmable logic 通过 AXI 真正搬运 DDR 数据，再测量 transaction granularity 如何改变 end-to-end DMA workload。

前置：LSN-018、LAB-HW-09 integrity PASS，以及已经工作的 PS/Linux + JTAG path。

**Project Trace:** RMD-014A · T-HW-010/T-HW-011

本章使用 AMD AXI Central Direct Memory Access（AXI CDMA），**不**要求从零手写完整 AXI master。

## 1. 和 HW-09 相比，真正变了什么？

HW-09 证明 PS/Linux system memory 能稳定保存 known bytes。

HW-10 改变的是 data mover：

```console
HW-09: CPU / Linux memory operations → DDR integrity

HW-10: PL AXI CDMA → S_AXI_HP0_FPD → DDR
```

这一次比较的是真正的 **PL AXI master path**。

但它仍然不是正式 FlyBrain DDR-backed synapse store。

## 2. 实体 data path

<svg xmlns="http://www.w3.org/2000/svg" width="1040" height="390" viewBox="0 0 1040 390" role="img" aria-label="LAB-HW-10 AXI CDMA control and DDR data paths">
  <rect x="25" y="70" width="155" height="80" rx="10" fill="#eef0ff" stroke="#333"/>
  <text x="102" y="103" text-anchor="middle" font-size="14">PS / Linux</text>
  <text x="102" y="127" text-anchor="middle" font-size="12">benchmark.py</text>
  <rect x="230" y="45" width="185" height="80" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="322" y="78" text-anchor="middle" font-size="14">M_AXI_HPM0_FPD</text>
  <text x="322" y="102" text-anchor="middle" font-size="12">control path</text>
  <rect x="470" y="45" width="180" height="80" rx="10" fill="#fff3cd" stroke="#333"/>
  <text x="560" y="78" text-anchor="middle" font-size="14">AXI CDMA</text>
  <text x="560" y="102" text-anchor="middle" font-size="12">S_AXI_LITE 0xA0020000</text>
  <rect x="470" y="220" width="180" height="80" rx="10" fill="#fff3cd" stroke="#333"/>
  <text x="560" y="253" text-anchor="middle" font-size="14">AXI CDMA M_AXI</text>
  <text x="560" y="277" text-anchor="middle" font-size="12">128 bit / burst ≤64</text>
  <rect x="700" y="220" width="150" height="80" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="775" y="253" text-anchor="middle" font-size="14">S_AXI_HP0_FPD</text>
  <text x="775" y="277" text-anchor="middle" font-size="12">non-coherent</text>
  <rect x="895" y="220" width="120" height="80" rx="10" fill="#fee" stroke="#333"/>
  <text x="955" y="253" text-anchor="middle" font-size="14">DDR4</text>
  <text x="955" y="277" text-anchor="middle" font-size="12">DMA buffer</text>
  <path d="M180 100 L230 85 M415 85 L470 85 M560 125 L560 220 M650 260 L700 260 M850 260 L895 260" stroke="#333" stroke-width="2" fill="none"/>
</svg>

control 与 data 是两条不同 path：

- PS 通过 `M_AXI_HPM0_FPD` 写 AXI CDMA register。
- 真正搬 DDR 数据时，CDMA 通过 `S_AXI_HP0_FPD` 成为 AXI master。

## 3. 为什么 DMA buffer 需要特殊 contract

AMD 把 `S_AXI_HP0_FPD` 定义为 **non-coherent** PL→DDR path。

所以普通 cached Python memory 不能直接假设为安全 DMA buffer。

physical mode 要求：

- `/dev/udmabuf0`；
- 至少 **2 MiB**；
- physical address 可从 `/sys/class/u-dma-buf/udmabuf0/phys_addr` 读取；
- helper 使用 `O_SYNC` 打开 device；
- 整个课程 buffer 必须位于 `HP0_DDR_LOW`。

课程把 upstream **u-dma-buf** 当作 buffer provider；不要求学生自己写或修改 kernel driver。

如果当前 Ubuntu/kernel image 无法满足这些条件，就停止并保存 failure evidence；不要改成猜一个 DDR physical address。

## 4. bytes 相同，只改变 transaction granularity

两种 pattern 都把完全相同的 **256 KiB** payload，从同一个 source region 搬到同一个 destination region。

### Contiguous

```console
1 request × 256 KiB
```

### Small/scattered

```console
1024 requests × 256 B
```

block order 是 deterministic：

```console
block = (257*i + 17) mod 1024
```

257 与 1024 互质，所以每个 block 恰好出现一次。

这里不需要 random seed。

## 5. timer 到底包住了什么？

timer 在 helper program 第一个 transfer 之前开始，在最后一次 DMA completion poll 之后结束。

所以 timing 包含：

- Python register write；
- Python status polling；
- AXI CDMA command overhead；
- AXI transfer；
- DDR path。

不包含 payload generation 与 post-run byte compare。

因此这是 **end-to-end software-controlled DMA workload**，不是纯 AXI bus analyzer measurement。

## 6. 上板前先 dry-run workload 与 statistics

```bash
python boards/kv260/runtime/axi_cdma_benchmark.py \
  --dry-run \
  --json-out /tmp/lab-hw-10-dry-run.json
```

dry-run 在 software memory model 上执行相同 block ordering 与 integrity logic。timing sample 使用 deterministic synthetic value，避免 CI 变成噪声很大的 performance test。

应看到：

```console
PRECHECK_INTEGRITY=PASS
MEASUREMENT_STABLE=1
PERFORMANCE_CONCLUSION_ALLOWED=1
STATUS=PASS
```

CI PASS 不是 physical bandwidth result。

## 7. 先证明两个 gate 都真的会挡住错误结论

integrity failure：

```bash
python boards/kv260/runtime/axi_cdma_benchmark.py \
  --dry-run --inject-corruption-for-test
```

应得到 `BENCHMARK_INTEGRITY_MISMATCH`，并禁止 performance conclusion。

stability failure：

```bash
python boards/kv260/runtime/axi_cdma_benchmark.py \
  --dry-run --unstable-for-test
```

应得到 `MEASUREMENT_UNSTABLE` 与 `PERFORMANCE_CONCLUSION_ALLOWED=0`。

连这两个 failure 都不会触发的 benchmark，不应该上真实板。

## 8. Build AXI CDMA bitstream

development host：

```bash
vivado -mode batch -nojournal \
  -log lab-hw-10-build.log \
  -source boards/kv260/scripts/build_lab10_axi_cdma.tcl
```

冻结 build facts：

- Simple DMA；
- Scatter/Gather off；
- 128-bit M_AXI；
- max burst 64；
- 64-bit address register；
- `S_AXI_HP0_FPD`；
- control base `0xA0020000`；
- 第一版 Lab 只映射 `HP0_DDR_LOW`；
- DRC 或负 setup/hold slack 会阻止 bitstream。

expected bitstream：

`build/kv260/lab-hw-10/kv260_axi_cdma_benchmark.bit`

## 9. program 时不要静默重初始化正在运行的 PS

保持 LAB-HW-05 的 Linux system 继续运行。如有 active Kria application，先 unload。

使用共享 direct-JTAG helper program PL：

```bash
vivado -mode batch -nojournal \
  -log lab-hw-10-program.log \
  -source boards/kv260/scripts/program_bitstream.tcl \
  -tclargs build/kv260/lab-hw-10/kv260_axi_cdma_benchmark.bit
```

**不要**为了强行跑通 Lab 而执行会 reset/reconfigure live Linux DDR subsystem 的 PS initialization sequence。

如果当前 boot firmware 没有让所需 HP0 runtime path 可用，就保存 failure，让 T-HW-010 保持 blocked，后续再修 platform flow。

## 10. 检查 DMA-safe buffer provider

physical benchmark 之前，runtime host 必须已经提供 u-dma-buf。

检查：

```bash
ls -l /dev/udmabuf0
cat /sys/class/u-dma-buf/udmabuf0/phys_addr
cat /sys/class/u-dma-buf/udmabuf0/size
```

required size：至少 **2097152 bytes**。

device 不存在时，应先用与当前 kernel 匹配的 upstream u-dma-buf 准备 course image。upstream driver 能分配 userspace-mappable contiguous DMA buffer，并在 sysfs 暴露 physical address。

graded run 过程中不要临时下载/编译一个未记录版本的 kernel module。

## 11. 运行真实 benchmark

把 `axi_cdma_benchmark.py` 复制到 PS/Linux，然后运行：

```bash
sudo python3 /tmp/axi_cdma_benchmark.py \
  --physical \
  --json-out /tmp/lab-hw-10-trace.json
```

physical mode 需要 root，因为 AXI CDMA control register 通过 fixed `/dev/mem` MMIO 访问。

helper 会拒绝：

- u-dma-buf 缺失；
- buffer 小于 2 MiB；
- buffer 不在 `HP0_DDR_LOW`；
- CDMA reset/timeout/error；
- data mismatch；
- 两批 measurement 不稳定。

只有两种 pattern 都通过 integrity 与 ≤10% stability gate 才能 PASS。

## 12. 读结果时不要过度推广

每种 pattern、每一批，checker 都保存 20 个 raw sample，以及：

- median；
- minimum；
- maximum；
- median MiB/s。

然后计算两批 median 的 relative difference。

只有两种 pattern 都 stable 时才打印：

`SCATTERED_TO_CONTIGUOUS_MEDIAN_TIME_RATIO=...`

这个 ratio 只能解释为：

> 在这一个 bitstream、256 KiB workload、Python polling boundary、AXI CDMA configuration 与 runtime session 下，两种 transaction pattern 观察到这些 end-to-end time。

不能改写成“DDR 快 N 倍”或“AXI burst 永远快 N 倍”。

## 13. Failure class

重要分类包括：

- `DMA_BUFFER_DEVICE_MISSING`
- `DMA_BUFFER_SYSFS_MISSING`
- `DMA_BUFFER_TOO_SMALL`
- `DMA_BUFFER_OUTSIDE_HP0_DDR_LOW`
- `TRANSPORT_REQUIRES_ROOT`
- `TRANSPORT_PERMISSION_OR_POLICY`
- `CDMA_RESET_TIMEOUT`
- `CDMA_TIMEOUT`
- `CDMA_ERROR`
- `BENCHMARK_INTEGRITY_MISMATCH`
- `POST_BENCHMARK_INTEGRITY_MISMATCH`
- `MEASUREMENT_UNSTABLE`

transport/platform failure、correctness failure、measurement-quality failure 是三类不同工程问题。

## 14. Expected Evidence / Save Evidence

保留：

- Git commit；
- board/carrier revision；
- Ubuntu/kernel identity；
- course image 实际使用的 u-dma-buf source/version identity；
- `/dev/udmabuf0` size / physical base；
- buffer cache-mode contract（`O_SYNC`）；
- LAB-HW-10 bitstream SHA-256；
- Vivado build、DRC、timing、utilization、program log；
- AXI CDMA configuration / control base；
- helper SHA-256；
- payload SHA-256；
- precheck / post-batch integrity result；
- warm-up count 与全部 measured raw sample；
- 每种 pattern 两批 median/min/max；
- stability percentage；
- 只有允许 conclusion 时才保存 ratio；
- `lab-hw-10-trace.json`；
- experiment date。

这会在教材/CI 层面闭合第一轮 Physical-Lab track，但真实 T-HW-010 仍需要实体板 evidence。

## 15. Human Check

请解释：

1. 为什么 control AXI 与 DDR data AXI 是不同 path？
2. 为什么 `S_AXI_HP0_FPD` 叫 non-coherent？
3. 为什么不能默认普通 cached Python memory 是 DMA-safe？
4. 为什么两种 pattern 都搬同样 256 KiB？
5. 为什么 small/scattered 使用 deterministic permutation？
6. timer 包含什么？
7. 为什么 5 次 warm-up 不进统计？
8. 为什么主结果用 median，而不是挑一个“最好”的 sample？
9. 为什么两批 drift >10% 就不能下结论？
10. 为什么最终 ratio 不是 peak-DDR claim？

## 16. 官方依据 / implementation notes

AMD sources：

- PG201：`S_AXI_HP0_FPD` 是 non-coherent PL→FPD/DDR path；HP interface 支持最高 128-bit data width。
- PG034：AXI CDMA 提供 memory-mapped AXI4 master + AXI4-Lite control；支持 Simple DMA；DataMover 会自动 partition burst，并处理 4 KiB boundary protection。
- PG034 register map：`CDMACR=0x00`、`CDMASR=0x04`、`SA=0x18`、`SA_MSB=0x1C`、`DA=0x20`、`DA_MSB=0x24`、`BTT=0x28`。
- PG034 throughput guidance：更大的 configured burst length 可以提高 realized CDMA bus utilization，但 system-level performance 仍取决于 workload。

buffer-provider basis：

- upstream u-dma-buf 文档说明了 userspace-mappable contiguous DMA buffer、sysfs physical-address exposure、Zynq UltraScale+ ARM64 support，以及通过 `O_SYNC` 控制 cache behavior。

本 Lab 只冻结一套 teaching configuration；不会把 u-dma-buf 或 AXI CDMA 升级成永久 FlyBrain architecture choice。